In [ ]:
!pip -q install google-cloud-bigquery google-cloud-pubsub

In [ ]:
# Import libraries
from google.cloud import bigquery, pubsub_v1
from google.api_core.exceptions import Conflict, NotFound
from datetime import datetime
import time

In [ ]:
# Set variables
PROJECT_ID = "qwiklabs-gcp-00-871084f9eb9e"
DATASET_ID = "flight_data"
TABLE_ID = "transponder_messages"

SOURCE_PROJECT = "paul-leroy"
TOPIC_ID = "flight-transponder"
SUBSCRIPTION_ID = "flight-transponder-sub"

bq_client = bigquery.Client(project=PROJECT_ID)
subscriber = pubsub_v1.SubscriberClient()

table_ref = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"
topic_path = f"projects/{SOURCE_PROJECT}/topics/{TOPIC_ID}"
subscription_path = subscriber.subscription_path(PROJECT_ID, SUBSCRIPTION_ID)

print(topic_path)
print(subscription_path)

projects/paul-leroy/topics/flight-transponder
projects/qwiklabs-gcp-00-871084f9eb9e/subscriptions/flight-transponder-sub


In [ ]:
# Create dataset
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = "US"

try:
    bq_client.create_dataset(dataset_ref)
    print(f"Created dataset {PROJECT_ID}.{DATASET_ID}")
except Conflict:
    print(f"Dataset {PROJECT_ID}.{DATASET_ID} already exists")

Created dataset qwiklabs-gcp-00-871084f9eb9e.flight_data


In [ ]:
# Create BigQuery table with the challenge schema
schema = [
    bigquery.SchemaField("MT", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("TT", "INT64", mode="NULLABLE"),
    bigquery.SchemaField("SID", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("AID", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("Hex", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("FID", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("DMG", "DATE", mode="NULLABLE"),
    bigquery.SchemaField("TMG", "TIME", mode="NULLABLE"),
    bigquery.SchemaField("DML", "DATE", mode="NULLABLE"),
    bigquery.SchemaField("TML", "TIME", mode="NULLABLE"),
    bigquery.SchemaField("CS", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("Alt", "INT64", mode="NULLABLE"),
    bigquery.SchemaField("GS", "INT64", mode="NULLABLE"),
    bigquery.SchemaField("Trk", "INT64", mode="NULLABLE"),
    bigquery.SchemaField("Lat", "FLOAT64", mode="NULLABLE"),
    bigquery.SchemaField("Lng", "FLOAT64", mode="NULLABLE"),
    bigquery.SchemaField("VR", "INT64", mode="NULLABLE"),
    bigquery.SchemaField("Sq", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("Alrt", "INT64", mode="NULLABLE"),
    bigquery.SchemaField("Emer", "INT64", mode="NULLABLE"),
    bigquery.SchemaField("SPI", "INT64", mode="NULLABLE"),
    bigquery.SchemaField("Gnd", "INT64", mode="NULLABLE"),
]

table = bigquery.Table(table_ref, schema=schema)

try:
    bq_client.create_table(table)
    print(f"Created table {table_ref}")
except Conflict:
    print(f"Table {table_ref} already exists")

Created table qwiklabs-gcp-00-871084f9eb9e.flight_data.transponder_messages


In [ ]:
# Create subscription to the source topic
try:
    subscriber.create_subscription(
        request={"name": subscription_path, "topic": topic_path}
    )
    print(f"Created subscription {subscription_path}")
except Conflict:
    print(f"Subscription {subscription_path} already exists")

Created subscription projects/qwiklabs-gcp-00-871084f9eb9e/subscriptions/flight-transponder-sub


In [ ]:
# Helper functions to parse messages
def parse_date(value):
    if not value:
        return None
    return datetime.strptime(value, "%Y/%m/%d").date().isoformat()

def parse_time(value):
    if not value:
        return None
    return datetime.strptime(value, "%H:%M:%S.%f").time().isoformat()

def to_int(value):
    if value == "" or value is None:
        return None
    return int(value)

def to_float(value):
    if value == "" or value is None:
        return None
    return float(value)

def parse_message(msg_text):
    parts = msg_text.strip().split(",")
    if len(parts) != 22:
        return None

    return {
        "MT": parts[0] or None,
        "TT": to_int(parts[1]),
        "SID": parts[2] or None,
        "AID": parts[3] or None,
        "Hex": parts[4] or None,
        "FID": parts[5] or None,
        "DMG": parse_date(parts[6]),
        "TMG": parse_time(parts[7]),
        "DML": parse_date(parts[8]),
        "TML": parse_time(parts[9]),
        "CS": parts[10] or None,
        "Alt": to_int(parts[11]),
        "GS": to_int(parts[12]),
        "Trk": to_int(parts[13]),
        "Lat": to_float(parts[14]),
        "Lng": to_float(parts[15]),
        "VR": to_int(parts[16]),
        "Sq": parts[17] or None,
        "Alrt": to_int(parts[18]),
        "Emer": to_int(parts[19]),
        "SPI": to_int(parts[20]),
        "Gnd": to_int(parts[21]),
    }

In [ ]:
# Pull messages for 3 minutes and write to BigQuery
end_time = time.time() + 180
inserted = 0

while time.time() < end_time:
    response = subscriber.pull(
        request={
            "subscription": subscription_path,
            "max_messages": 200,
        },
        timeout=30,
    )

    ack_ids = []
    rows_to_insert = []

    for received_message in response.received_messages:
        msg_text = received_message.message.data.decode("utf-8")

        for line in msg_text.splitlines():
            line = line.strip()
            if not line:
                continue

            row = parse_message(line)
            if row is not None:
                rows_to_insert.append(row)

        ack_ids.append(received_message.ack_id)

    if rows_to_insert:
        errors = bq_client.insert_rows_json(table_ref, rows_to_insert)
        if not errors:
            subscriber.acknowledge(
                request={"subscription": subscription_path, "ack_ids": ack_ids}
            )
            inserted += len(rows_to_insert)
            print(f"Inserted {len(rows_to_insert)} rows, total inserted: {inserted}")
        else:
            print("Insert errors:", errors)

    time.sleep(2)

print(f"Finished. Total inserted rows: {inserted}")

Inserted 200 rows, total inserted: 200
Inserted 200 rows, total inserted: 400
Inserted 200 rows, total inserted: 600
Inserted 200 rows, total inserted: 800
Inserted 200 rows, total inserted: 1000
Inserted 200 rows, total inserted: 1200
Inserted 200 rows, total inserted: 1400
Inserted 200 rows, total inserted: 1600
Inserted 200 rows, total inserted: 1800
Inserted 200 rows, total inserted: 2000
Inserted 140 rows, total inserted: 2140
Inserted 192 rows, total inserted: 2332
Inserted 200 rows, total inserted: 2532
Inserted 200 rows, total inserted: 2732
Inserted 200 rows, total inserted: 2932
Inserted 200 rows, total inserted: 3132
Inserted 200 rows, total inserted: 3332
Inserted 2 rows, total inserted: 3334
Inserted 200 rows, total inserted: 3534
Inserted 200 rows, total inserted: 3734
Inserted 200 rows, total inserted: 3934
Inserted 200 rows, total inserted: 4134
Inserted 200 rows, total inserted: 4334
Inserted 200 rows, total inserted: 4534
Inserted 200 rows, total inserted: 4734
Insert

In [ ]:
# Count records in the BigQuery table
count_sql = f"""
SELECT COUNT(*) AS total_records
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
"""
bq_client.query(count_sql).to_dataframe()

,total_records
0,13840


In [ ]:
# Query locations for GeoViz using GEOGRAPHY point
geoviz_sql = f"""
SELECT
  Hex,
  CS,
  Alt,
  GS,
  Trk,
  ST_GEOGPOINT(Lng, Lat) AS point
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
WHERE Lat IS NOT NULL
  AND Lng IS NOT NULL
"""
locations_df = bq_client.query(geoviz_sql).to_dataframe()
locations_df.head(20)

,Hex,CS,Alt,GS,Trk,point
0,4CAC84,None,35025,<NA>,<NA>,POINT(-0.68932 51.446)
1,4D243C,None,9275,<NA>,<NA>,POINT(-0.91647 51.6286)
2,899038,None,6750,<NA>,<NA>,POINT(-0.02695 51.56602)
3,3980E0,None,29000,<NA>,<NA>,POINT(0.04868 51.43184)
4,4BB26B,None,6250,<NA>,<NA>,POINT(-0.5349 51.6079)
5,3C55C3,None,5600,<NA>,<NA>,POINT(-0.65941 51.54078)
6,485A33,None,40000,<NA>,<NA>,POINT(-0.18112 51.30001)
7,4869C3,None,35000,<NA>,<NA>,POINT(0.35416 51.44888)
8,4009D9,None,4550,<NA>,<NA>,POINT(-0.62386 51.45731)
9,407B8B,None,21100,<NA>,<NA>,POINT(-0.73219 51.155)


In [20]:
print(geoviz_sql)



SELECT
  Hex,
  CS,
  Alt,
  GS,
  Trk,
  ST_GEOGPOINT(Lng, Lat) AS point
FROM `qwiklabs-gcp-00-871084f9eb9e.flight_data.transponder_messages`
WHERE Lat IS NOT NULL
  AND Lng IS NOT NULL

